In [0]:
# ============================================================
# CELL 1: ADLS Connection + Config
# ============================================================

storage_account_name = "stsapadlsrawdev"
storage_account_key  = "YOUR_STORAGE_KEY_HERE"
gold_container       = "gold"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

spark.conf.set("spark.databricks.io.cache.enabled",               "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",    "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",      "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print("ADLS connection configured!")
print("Delta Cache enabled!")
print("Auto Optimize enabled!")
print("Schema Evolution enabled!")

ADLS connection configured!


In [0]:
# ============================================================
# CELL 2: Define Paths
# ============================================================

storage_account_name  = "stsapadlsrawdev"
gold_container        = "gold"
gold_path             = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

po_filtered_path      = f"{gold_path}/po_filtered/"
po_price_audit_path   = f"{gold_path}/ptp_test_1/"
watermark_path        = f"{gold_path}/ptp_test_1_watermark/"

print(f"Gold po_filtered    : {po_filtered_path}")
print(f"Gold ptp_test_1     : {po_price_audit_path}")
print(f"Watermark           : {watermark_path}")

Gold po_filtered    : abfss://gold@stsapadlsrawdev.dfs.core.windows.net/po_filtered/
Gold ptp_test_1     : abfss://gold@stsapadlsrawdev.dfs.core.windows.net/ptp_test_1/
Watermark           : abfss://gold@stsapadlsrawdev.dfs.core.windows.net/ptp_test_1_watermark/


In [0]:
# ============================================================
# CELL 3: Setup Watermark (first run only)
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col, current_timestamp, max as spark_max, \
    min as spark_min, countDistinct, round as spark_round, \
    dense_rank, lit
from pyspark.sql.window import Window

watermark_exists = DeltaTable.isDeltaTable(spark, watermark_path)

if not watermark_exists:
    schema = StructType([
        StructField("table_name",           StringType(), False),
        StructField("last_processed_month", StringType(), True)
    ])

    df_wm = spark.createDataFrame([
        ("ptp_test_1", None)
    ], schema)

    df_wm.write.format("delta").mode("overwrite").save(watermark_path)
    print("Watermark table created!")
else:
    print("Watermark table already exists!")

# Read current watermark
last_processed_month = spark.read.format("delta") \
    .load(watermark_path) \
    .filter(col("table_name") == "ptp_test_1") \
    .collect()[0]["last_processed_month"]

print(f"Last processed month: {last_processed_month}")

Watermark table already exists!
Last processed month: None-None


In [0]:
# ============================================================
# CELL 4: Read po_filtered (with reprocessing window)
# ============================================================

from datetime import datetime
from dateutil.relativedelta import relativedelta

df_po_filtered = spark.read \
    .format("delta") \
    .load(po_filtered_path)

# Apply reprocessing window
if last_processed_month is not None and last_processed_month != 'None-None':
    last_month      = datetime.strptime(last_processed_month, "%Y-%m")
    reprocess_from  = last_month - relativedelta(months=2)
    reprocess_year  = reprocess_from.year
    reprocess_month = reprocess_from.month

    df_po_filtered = df_po_filtered.filter(
        (col("po_year") > reprocess_year) |
        (
            (col("po_year")  == reprocess_year) &
            (col("po_month") >= reprocess_month)
        )
    )
    print(f"Processing from: {reprocess_from.strftime('%Y-%m')}")
else:
    print("First run - processing all data!")

print(f"Records to process: {df_po_filtered.count()}")

First run - processing all data!
Records to process: 3
+----------+-----------+---------+----------+---------+-------+--------+-----------+------------+--------------+----------------+----------------+--------------+-----------------+-------------+------------+---------------+--------------------+---------------+--------+---------+----------+-----+---------+--------------+-------------+-------------+----------------+-------+--------+--------------------+--------------------+--------------------+--------------------+
| po_number|vendor_name|vendor_id|   po_date|po_status|po_type|currency|po_category|company_code|purchasing_org|purchasing_group|release_strategy|release_status|release_indicator|payment_terms|po_line_item|material_number|material_description|unit_of_measure|quantity|net_price|price_unit|plant|net_value|material_group|deletion_flag|item_category|storage_location|po_year|po_month|  source_recordstamp|    silver_load_time|              po_key|      gold_load_time|
+----------

In [0]:
# ============================================================
# CELL 5: Calculate Price Stats per PO_KEY
# Equivalent to TEMP_P2P_1_2 in your SQL
# ============================================================

# Step 1: Calculate min price and count distinct prices per PO_KEY
df_price_stats = df_po_filtered \
    .groupBy("po_key") \
    .agg(
        countDistinct("net_price").alias("count_po_price"),
        spark_min("net_price").alias("min_po_price")
    ) \
    .filter(col("count_po_price") > 1)

print(f"PO_KEYs with price variance: {df_price_stats.count()}")
df_price_stats.show()

# Step 2: Join back to get all line items
df_price_variance = df_po_filtered.alias("y").join(
    df_price_stats.alias("x"),
    on  = "po_key",
    how = "inner"
).select(
    col("y.po_key"),
    col("x.count_po_price"),
    col("x.min_po_price"),
    col("y.po_number"),
    col("y.po_line_item"),
    col("y.po_month"),
    col("y.po_year"),
    col("y.vendor_id"),
    col("y.vendor_name"),
    col("y.material_number"),
    col("y.material_description"),
    col("y.quantity"),
    col("y.net_price"),
    col("y.currency"),
    col("y.po_date"),
    col("y.deletion_flag"),
    col("y.release_status"),
    col("y.po_status"),
    col("y.plant"),
    col("y.company_code"),
    col("y.purchasing_org"),
    col("y.purchasing_group"),
    (col("y.net_price") - col("x.min_po_price")).alias("price_diff"),
    spark_round(
        (col("y.net_price") - col("x.min_po_price")) / col("x.min_po_price"),
        3
    ).alias("price_variance")
)

print(f"Price variance records: {df_price_variance.count()}")
df_price_variance.show(5)

PO_KEYs with price variance: 1
+--------------------+--------------+------------+
|              po_key|count_po_price|min_po_price|
+--------------------+--------------+------------+
|C0000003120221000...|             2|         5.0|
+--------------------+--------------+------------+

Price variance records: 2
+--------------------+--------------+------------+----------+------------+--------+-------+---------+-----------+---------------+--------------------+--------+---------+--------+----------+-------------+--------------+---------+-----+------------+--------------+----------------+----------+--------------+
|              po_key|count_po_price|min_po_price| po_number|po_line_item|po_month|po_year|vendor_id|vendor_name|material_number|material_description|quantity|net_price|currency|   po_date|deletion_flag|release_status|po_status|plant|company_code|purchasing_org|purchasing_group|price_diff|price_variance|
+--------------------+--------------+------------+----------+------------+-

In [0]:
# ============================================================
# CELL 6: Final Audit Output + CASE_ID
# Equivalent to O_P2P_1 in your SQL
# ============================================================

# Step 1: Get max variance per PO_KEY
df_max_variance = df_price_variance \
    .groupBy("po_key") \
    .agg(spark_max("price_variance").alias("max_variance")) \
    .filter(col("max_variance") >= 0.05)

print(f"PO_KEYs with >= 5% variance: {df_max_variance.count()}")

# Step 2: Join back to get all line items
df_price_audit = df_price_variance.alias("b").join(
    df_max_variance.alias("a"),
    on  = "po_key",
    how = "inner"
).select(
    col("b.po_key"),
    col("b.count_po_price"),
    col("b.min_po_price"),
    col("b.po_number"),
    col("b.po_line_item"),
    col("b.po_month"),
    col("b.po_year"),
    col("b.vendor_id"),
    col("b.vendor_name"),
    col("b.material_number"),
    col("b.material_description"),
    col("b.quantity"),
    col("b.net_price"),
    col("b.currency"),
    col("b.po_date"),
    col("b.deletion_flag"),
    col("b.release_status"),
    col("b.po_status"),
    col("b.plant"),
    col("b.company_code"),
    col("b.purchasing_org"),
    col("b.purchasing_group"),
    col("b.price_diff"),
    col("b.price_variance"),
    col("a.max_variance"),
    current_timestamp().alias("gold_load_time")
)

# Step 3: Add CASE_ID using dense_rank
window_case = Window.orderBy("po_key")
df_price_audit = df_price_audit \
    .withColumn("case_id", dense_rank().over(window_case))

print(f"Final audit records: {df_price_audit.count()}")
df_price_audit.orderBy(col("price_variance").desc()).show(5)

PO_KEYs with >= 5% variance: 1
Final audit records: 2
+--------------------+--------------+------------+----------+------------+--------+-------+---------+-----------+---------------+--------------------+--------+---------+--------+----------+-------------+--------------+---------+-----+------------+--------------+----------------+----------+--------------+------------+--------------------+-------+
|              po_key|count_po_price|min_po_price| po_number|po_line_item|po_month|po_year|vendor_id|vendor_name|material_number|material_description|quantity|net_price|currency|   po_date|deletion_flag|release_status|po_status|plant|company_code|purchasing_org|purchasing_group|price_diff|price_variance|max_variance|      gold_load_time|case_id|
+--------------------+--------------+------------+----------+------------+--------+-------+---------+-----------+---------------+--------------------+--------+---------+--------+----------+-------------+--------------+---------+-----+------------+---

In [0]:
# ============================================================
# CELL 7: MERGE into ptp_test_1 Delta
# ============================================================

price_audit_exists = DeltaTable.isDeltaTable(spark, po_price_audit_path)

if not price_audit_exists:
    df_price_audit.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("po_year", "po_month") \
        .option("mergeSchema", "true") \
        .save(po_price_audit_path)

    spark.sql(f"""
        ALTER TABLE delta.`{po_price_audit_path}`
        SET TBLPROPERTIES (
            'delta.dataSkippingNumIndexedCols' = '32',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    print("ptp_test_1 Delta table created!")
else:
    delta_audit = DeltaTable.forPath(spark, po_price_audit_path)

    delta_audit.alias("target").merge(
        df_price_audit.alias("source"),
        """
        target.po_number    = source.po_number AND
        target.po_line_item = source.po_line_item AND
        target.po_year      = source.po_year     AND
        target.po_month     = source.po_month
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("ptp_test_1 MERGE completed!")

total = spark.read.format("delta").load(po_price_audit_path).count()
print(f"Total records in ptp_test_1: {total}")

ptp_test_1 Delta table created!
Total records in ptp_test_1: 2


In [0]:
# ============================================================
# CELL 8: Update Watermark
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType

latest_year  = df_price_audit.agg(spark_max("po_year")).collect()[0][0]
latest_month = df_price_audit.agg(spark_max("po_month")).collect()[0][0]

if latest_year is not None and latest_month is not None:
    latest_ym = f"{latest_year}-{str(latest_month).zfill(2)}"
    print(f"Latest month processed: {latest_ym}")

    schema = StructType([
        StructField("table_name",           StringType(), False),
        StructField("last_processed_month", StringType(), True)
    ])

    delta_wm = DeltaTable.forPath(spark, watermark_path)
    delta_wm.alias("target").merge(
        spark.createDataFrame(
            [("ptp_test_1", latest_ym)], schema
        ).alias("source"),
        "target.table_name = source.table_name"
    ).whenMatchedUpdateAll() \
     .execute()

    print("Watermark updated!")
else:
    print("No audit cases found — watermark unchanged!")

spark.read.format("delta").load(watermark_path).show()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7810619990571081>, line 7
      1 # ============================================================
      2 # CELL 8: Update Watermark
      3 # ============================================================
      5 from pyspark.sql.types import StructType, StructField, StringType
----> 7 latest_year  = df_price_audit.agg(spark_max("po_year")).collect()[0][0]
      8 latest_month = df_price_audit.agg(spark_max("po_month")).collect()[0][0]
     10 if latest_year is not None and latest_month is not None:

NameError: name 'df_price_audit' is not defined

In [0]:
# ============================================================
# CELL 9: OPTIMIZE + Z-Order + VACUUM
# ============================================================

delta_audit = DeltaTable.forPath(spark, po_price_audit_path)

delta_audit.optimize().executeZOrderBy(
    "vendor_id",
    "material_number",
    "po_key"
)
print("OPTIMIZE + Z-Order completed!")

delta_audit.vacuum(168)
print("VACUUM completed!")

OPTIMIZE + Z-Order completed!
VACUUM completed!


In [0]:
# ============================================================
# CELL 10: Final Verification
# ============================================================

print("\n=== ptp_test_1 Final Summary ===")

# Watermark
print("\n-- Watermark --")
spark.read.format("delta").load(watermark_path).show()

# Record count
total = spark.read.format("delta").load(po_price_audit_path).count()
print(f"Total audit cases: {total}")

# Final audit output
print("\n-- Audit Cases (ordered by variance) --")
spark.read.format("delta") \
    .load(po_price_audit_path) \
    .select(
        "case_id",
        "po_number",
        "po_line_item",
        "material_number",
        "material_description",
        "vendor_id",
        "net_price",
        "min_po_price",
        "price_diff",
        "price_variance",
        "max_variance",
        "po_year",
        "po_month"
    ) \
    .orderBy(col("price_variance").desc()) \
    .show(10)

# Partitions
print("\n-- Partitions --")
files = dbutils.fs.ls(
    f"abfss://gold@{storage_account_name}.dfs.core.windows.net/ptp_test_1/"
)
for f in files:
    print(f.path)


=== ptp_test_1 Final Summary ===

-- Watermark --
+----------+--------------------+
|table_name|last_processed_month|
+----------+--------------------+
|ptp_test_1|             2022-01|
+----------+--------------------+

Total audit cases: 2

-- Audit Cases (ordered by variance) --
+-------+----------+------------+---------------+--------------------+---------+---------+------------+----------+--------------+------------+-------+--------+
|case_id| po_number|po_line_item|material_number|material_description|vendor_id|net_price|min_po_price|price_diff|price_variance|max_variance|po_year|po_month|
+-------+----------+------------+---------------+--------------------+---------+---------+------------+----------+--------------+------------+-------+--------+
|      1|4500000001|          10|       C0000003|Synthetic Dish wa...| CV000001|     10.0|         5.0|       5.0|           1.0|         1.0|   2022|       1|
|      1|4500000000|          10|       C0000003|Synthetic Dish wa...| CV000